# Integrated Pipeline Testing

Using 4 demo documents linked at \
https://en.wikipedia.org/wiki/Fuxing_(train) \
https://en.wikipedia.org/wiki/Hexie_(train) \
https://en.wikipedia.org/wiki/MTR_CRRC_Changchun_EMU \
https://en.wikipedia.org/wiki/MTR_SP1900_EMU

In [1]:
%cd Multi-modal-RAG

/content/Multi-modal-RAG


In [2]:
import transformers

transformers.__version__

'4.57.6'

## Imports

In [3]:
from config.settings import RAGConfig
from RAGPipeline import RAGSystem

/usr/local/lib/python3.12/dist-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''
/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for th

## Initialization and Setup

With dense retrieval mode

In [4]:
config = RAGConfig(collection_name="integration_test")
rag_system = RAGSystem(config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Image store initialized at: ./image_store


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2589: UserWarning: for cls_token: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict

Multi-modal embedding model (DINOv2+Talk2DINO) loaded.
Multi-vector database loaded.


## Ingest Documents

In [5]:
rag_system.ingest_documents("./documents/")

Stored image MTR CRRC Changchun EMU - Wikipedia.pdf_1_9a12bc4cf514d795e63ec62c05cd6268 (103801 bytes)
Stored image MTR CRRC Changchun EMU - Wikipedia.pdf_1_39a225b0b7d6bd718a9f282f5c994bd1 (78628 bytes)
Stored image MTR CRRC Changchun EMU - Wikipedia.pdf_3_7f0c9c878d6faea017fa40086d0f4a48 (75974 bytes)
Stored image MTR CRRC Changchun EMU - Wikipedia.pdf_4_479835ecfa1f1b3be7301007a1396ddf (96169 bytes)
Stored image MTR CRRC Changchun EMU - Wikipedia.pdf_4_cb311fe37a9cb074c196cda60331ef7e (62392 bytes)
Processed MTR CRRC Changchun EMU - Wikipedia.pdf: 8 chunks
5 images extracted
Stored image Hexie (train) - Wikipedia.pdf_1_f80b6adf3c2d2ef6e7e9de944662ebed (81100 bytes)
Stored image Hexie (train) - Wikipedia.pdf_3_c61c7f122640dd009275adbbda41416b (34152 bytes)
Stored image Hexie (train) - Wikipedia.pdf_3_cb5c05d0dfa34eb1b11cdeb861ce7c68 (65305 bytes)
Stored image Hexie (train) - Wikipedia.pdf_17_dad7c95b707591187f28309aa9014931 (24581 bytes)
Stored image Hexie (train) - Wikipedia.pdf_17_1

## Load Question Dataset

In [6]:
import json

with open("./questions_annotated.json", "r") as f:
    dataset = json.load(f)

dataset[0]

{'idx': 0,
 'question': 'What colors have been used on the livery of Fuxing trains?',
 'answers': ['Blue and white (non-standard)',
  'Red, brown and silver',
  'Red and silver',
  'Purple (Asian games specail livery)',
  'Green (on CR200J)']}

## Query and Generation

In [13]:
from datetime import datetime
import os
import time
from tqdm import tqdm
from typing import List, Dict


def generate_benchmark_answers_sync(
        rag_system: RAGSystem,
        dataset: List[Dict],
        output_path: str = "./results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the dataset.

    args:
    - rag_system (RAGSystem): Initialized RAGSystem instance
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []

    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")

    total = len(dataset)

    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue

            try:
                question = item.get('question')
                qid = item.get('qid', idx)
                print(f"Question: {question}")

                rag_result = rag_system.query(question)

                result = {
                    'qid': qid,
                    'question': question,
                    'answer': rag_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': rag_result.get('retrieval_mode'),
                        'documents_retrieved': rag_result.get('retrieval_metadata', {}).get('documents_retrieved', 0),
                        'images_used': rag_result.get('generation_metadata', {}).get('images_used', 0),
                    }
                }

                if 'answers' in item:
                    result['reference_answers'] = item['answers']

                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')

                results.append(result)

            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")

            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)

    print(f"\nCompleted! Total processed: {len(results)}")

    return results

In [29]:
from utils.benchmark_eval_helpers import grade_with_llm_judge

GPT-4o-mini, with multimodal retrieval; $k$=1

In [32]:
rag_system.config.top_k = 1

results_k1 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k1.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 2 images via multi-modal search
Using 2 images as context


Generating answers:   9%|▉         | 1/11 [00:10<01:41, 10.11s/it]

Question: What is the CR450 and what is special about it?
Retrieved 2 images via multi-modal search
Using 2 images as context


Generating answers:  18%|█▊        | 2/11 [00:19<01:28,  9.86s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 2 images via multi-modal search
Using 2 images as context


Generating answers:  91%|█████████ | 10/11 [01:24<00:08,  8.43s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 2 images via multi-modal search
Using 2 images as context


Generating answers: 100%|██████████| 11/11 [01:32<00:00,  8.38s/it]


Completed! Total processed: 11


In [33]:
grading_data_k1 = []
for result in results_k1:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k1.append(item)

grading_results_k1 = grade_with_llm_judge(
    responses=grading_data_k1,
    client=rag_system.generator.llm,
    output_file="./grading_results_k1.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k1['accuracy']:.2%}")
print(f"Correct: {grading_results_k1['correct_count']}/{grading_results_k1['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k1.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:21<00:00,  1.94s/it]


Detailed results saved to ./grading_results_k1.json

Grading Summary
Accuracy: 45.45%
Correct: 5/11

Detailed results saved to: ./grading_results_k1.json


GPT-4o-mini, with multimodal retrieval; $k$=3

In [34]:
rag_system.config.top_k = 3

results_k3 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k3.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 6 images via multi-modal search
Using 6 images as context


Generating answers:   9%|▉         | 1/11 [00:10<01:40, 10.09s/it]

Question: What is the CR450 and what is special about it?
Retrieved 6 images via multi-modal search
Using 6 images as context


Generating answers:  18%|█▊        | 2/11 [00:20<01:33, 10.44s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 6 images via multi-modal search
Using 6 images as context


Generating answers:  91%|█████████ | 10/11 [01:39<00:10, 10.27s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 6 images via multi-modal search
Using 6 images as context


Generating answers: 100%|██████████| 11/11 [01:48<00:00,  9.85s/it]


Completed! Total processed: 11


In [35]:
grading_data_k3 = []
for result in results_k3:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k3.append(item)

grading_results_k3 = grade_with_llm_judge(
    responses=grading_data_k3,
    client=rag_system.generator.llm,
    output_file="./grading_results_k3.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k3['accuracy']:.2%}")
print(f"Correct: {grading_results_k3['correct_count']}/{grading_results_k3['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k3.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:17<00:00,  1.55s/it]


Detailed results saved to ./grading_results_k3.json

Grading Summary
Accuracy: 36.36%
Correct: 4/11

Detailed results saved to: ./grading_results_k3.json


GPT-4o-mini, with multimodal retrieval; $k$=5

In [36]:
rag_system.config.top_k = 5

results_k5 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k5.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 10 images via multi-modal search
Using 10 images as context


Generating answers:   9%|▉         | 1/11 [00:10<01:44, 10.44s/it]

Question: What is the CR450 and what is special about it?
Retrieved 8 images via multi-modal search
Using 8 images as context


Generating answers:  18%|█▊        | 2/11 [00:21<01:37, 10.87s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 10 images via multi-modal search
Using 10 images as context


Generating answers:  91%|█████████ | 10/11 [01:41<00:10, 10.53s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 10 images via multi-modal search
Using 10 images as context


Generating answers: 100%|██████████| 11/11 [01:54<00:00, 10.37s/it]


Completed! Total processed: 11


In [37]:
grading_data_k5 = []
for result in results_k5:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k5.append(item)

grading_results_k5 = grade_with_llm_judge(
    responses=grading_data_k5,
    client=rag_system.generator.llm,
    output_file="./grading_results_k5.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k5['accuracy']:.2%}")
print(f"Correct: {grading_results_k5['correct_count']}/{grading_results_k5['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k5.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:20<00:00,  1.90s/it]


Detailed results saved to ./grading_results_k5.json

Grading Summary
Accuracy: 45.45%
Correct: 5/11

Detailed results saved to: ./grading_results_k5.json


GPT-4o-mini, with multimodal retrieval; $k$=7

In [38]:
rag_system.config.top_k = 7

results_k7 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k7.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 13 images via multi-modal search
Using 13 images as context


Generating answers:   9%|▉         | 1/11 [00:09<01:36,  9.67s/it]

Question: What is the CR450 and what is special about it?
Retrieved 12 images via multi-modal search
Using 12 images as context


Generating answers:  18%|█▊        | 2/11 [00:20<01:33, 10.37s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 13 images via multi-modal search
Using 13 images as context


Generating answers:  91%|█████████ | 10/11 [01:48<00:11, 11.44s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 14 images via multi-modal search
Using 14 images as context


Generating answers: 100%|██████████| 11/11 [01:59<00:00, 10.89s/it]


Completed! Total processed: 11


In [39]:
grading_data_k7 = []
for result in results_k7:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k7.append(item)

grading_results_k7 = grade_with_llm_judge(
    responses=grading_data_k7,
    client=rag_system.generator.llm,
    output_file="./grading_results_k7.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k7['accuracy']:.2%}")
print(f"Correct: {grading_results_k7['correct_count']}/{grading_results_k7['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k7.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:17<00:00,  1.62s/it]


Detailed results saved to ./grading_results_k7.json

Grading Summary
Accuracy: 45.45%
Correct: 5/11

Detailed results saved to: ./grading_results_k7.json


GPT-4o-mini, with multimodal retrieval; $k$=10

In [40]:
rag_system.config.top_k = 10

results_k10 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k10.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 19 images via multi-modal search
Using 19 images as context


Generating answers:   9%|▉         | 1/11 [00:11<01:57, 11.75s/it]

Question: What is the CR450 and what is special about it?
Retrieved 18 images via multi-modal search
Using 18 images as context


Generating answers:  18%|█▊        | 2/11 [00:24<01:52, 12.54s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 18 images via multi-modal search
Using 18 images as context


Generating answers:  91%|█████████ | 10/11 [02:06<00:13, 13.23s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 19 images via multi-modal search
Using 19 images as context


Generating answers: 100%|██████████| 11/11 [02:21<00:00, 12.83s/it]


Completed! Total processed: 11


In [41]:
grading_data_k10 = []
for result in results_k10:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k10.append(item)

grading_results_k10 = grade_with_llm_judge(
    responses=grading_data_k10,
    client=rag_system.generator.llm,
    output_file="./grading_results_k10.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k10['accuracy']:.2%}")
print(f"Correct: {grading_results_k10['correct_count']}/{grading_results_k10['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k10.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:16<00:00,  1.52s/it]


Detailed results saved to ./grading_results_k10.json

Grading Summary
Accuracy: 45.45%
Correct: 5/11

Detailed results saved to: ./grading_results_k10.json


GPT-4o-mini, with multimodal retrieval; $k$=15

In [42]:
rag_system.config.top_k = 15

results_k15 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k15.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 28 images via multi-modal search
Using 28 images as context


Generating answers:   9%|▉         | 1/11 [00:14<02:25, 14.59s/it]

Question: What is the CR450 and what is special about it?
Retrieved 28 images via multi-modal search
Using 28 images as context


Generating answers:  18%|█▊        | 2/11 [00:28<02:05, 13.97s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 27 images via multi-modal search
Using 27 images as context


Generating answers:  91%|█████████ | 10/11 [02:21<00:14, 14.33s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 28 images via multi-modal search
Using 28 images as context


Generating answers: 100%|██████████| 11/11 [02:38<00:00, 14.37s/it]


Completed! Total processed: 11


In [43]:
grading_data_k15 = []
for result in results_k15:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k15.append(item)

grading_results_k15 = grade_with_llm_judge(
    responses=grading_data_k15,
    client=rag_system.generator.llm,
    output_file="./grading_results_k15.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k15['accuracy']:.2%}")
print(f"Correct: {grading_results_k15['correct_count']}/{grading_results_k15['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k15.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:17<00:00,  1.61s/it]


Detailed results saved to ./grading_results_k15.json

Grading Summary
Accuracy: 45.45%
Correct: 5/11

Detailed results saved to: ./grading_results_k15.json


GPT-4o-mini, with multimodal retrieval; $k$=20

In [44]:
rag_system.config.top_k = 20

results_k20 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./results_k20.jsonl",
    rate_limit_delay=1.0,
    resume=False
)

Generating answers:   0%|          | 0/11 [00:00<?, ?it/s]

Question: What colors have been used on the livery of Fuxing trains?
Retrieved 36 images via multi-modal search
Using 36 images as context


Generating answers:   9%|▉         | 1/11 [00:17<02:52, 17.25s/it]

Question: What is the CR450 and what is special about it?
Retrieved 35 images via multi-modal search
Using 35 images as context


Generating answers:  18%|█▊        | 2/11 [00:35<02:41, 17.90s/it]

...

Question: What differences do first-class carriages of the SP1900 train have as compared to other carriages?
Retrieved 34 images via multi-modal search
Using 34 images as context


Generating answers:  91%|█████████ | 10/11 [02:38<00:15, 15.33s/it]

Question: What major incidents have happened to SP1900 trains?
Retrieved 35 images via multi-modal search
Using 35 images as context


Generating answers: 100%|██████████| 11/11 [02:53<00:00, 15.77s/it]


Completed! Total processed: 11


In [45]:
grading_data_k20 = []
for result in results_k20:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth
    }
    grading_data_k20.append(item)

grading_results_k20 = grade_with_llm_judge(
    responses=grading_data_k20,
    client=rag_system.generator.llm,
    output_file="./grading_results_k20.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k20['accuracy']:.2%}")
print(f"Correct: {grading_results_k20['correct_count']}/{grading_results_k20['total_count']}")
print(f"\nDetailed results saved to: ./grading_results_k20.json")


Grading 11 generated responses using LLM judge...


Grading: 100%|██████████| 11/11 [00:16<00:00,  1.52s/it]


Detailed results saved to ./grading_results_k20.json

Grading Summary
Accuracy: 54.55%
Correct: 6/11

Detailed results saved to: ./grading_results_k20.json
